In [ ]:
# Phase 2 laterality verify: the census (knee-phase2-laterality-census) read only
# the FIRST sorted instance per series to resolve laterality. Laterality (0020,0060)
# is documented as a series-level DICOM attribute, so it should be identical across
# every instance in a series -- but ImageLaterality matched 0/4407 studies in the
# census, which is suspicious enough to check whether tag survival is actually
# uniform across instances (e.g. an anonymization pipeline that didn't strip/keep
# tags consistently file-by-file). This checks EVERY instance, not just the first,
# for 50 studies the census resolved as "unknown" -- if none of them carry the tag
# on any instance, the 48.3% unknown figure is a real property of the data, not an
# artifact of only sampling one file per series.
import glob, os, shutil, sys

src_candidates = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)
comp_candidates = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)
SRC = src_candidates[0]
COMP_DIR = comp_candidates[0]

PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')

from knee.dicom import read_laterality_header
print('knee package imported successfully from', PKG)


In [ ]:
UNKNOWN_SAMPLE_UIDS = [
  "1.2.826.0.1.3680043.8.498.11926412946593048423086380882538180028",
  "1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967",
  "1.2.826.0.1.3680043.8.498.91385694666101904780012335954322545042",
  "1.2.826.0.1.3680043.8.498.12100254887364913022980951672127944515",
  "1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645",
  "1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166",
  "1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766",
  "1.2.826.0.1.3680043.8.498.76093766204928582819565636658977147523",
  "1.2.826.0.1.3680043.8.498.74160604730133050784449689952695940558",
  "1.2.826.0.1.3680043.8.498.76651080491383110752977286124044311636",
  "1.2.826.0.1.3680043.8.498.35961087303991442751747064990497975939",
  "1.2.826.0.1.3680043.8.498.11515987024418648685676369132061127664",
  "1.2.826.0.1.3680043.8.498.14490539215064287912254286565666891557",
  "1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678",
  "1.2.826.0.1.3680043.8.498.58739535411760425418941874065260802744",
  "1.2.826.0.1.3680043.8.498.12719455674338091089495020475033082405",
  "1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278",
  "1.2.826.0.1.3680043.8.498.62383135836496954297324866452887364097",
  "1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991",
  "1.2.826.0.1.3680043.8.498.41036022211588451079713314564568887994",
  "1.2.826.0.1.3680043.8.498.79991838737793096019496013540529642900",
  "1.2.826.0.1.3680043.8.498.11045459770905584149950259201414074545",
  "1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681",
  "1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250",
  "1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962",
  "1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768",
  "1.2.826.0.1.3680043.8.498.10417042894702157309437002434676100667",
  "1.2.826.0.1.3680043.8.498.99441773159476751698123795913150660766",
  "1.2.826.0.1.3680043.8.498.77299727102400166279263320763707637485",
  "1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144",
  "1.2.826.0.1.3680043.8.498.11201741528059502111642265469347871878",
  "1.2.826.0.1.3680043.8.498.48068031587855513429324313927339963541",
  "1.2.826.0.1.3680043.8.498.10759288702627176691392048851622294054",
  "1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910",
  "1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283",
  "1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266",
  "1.2.826.0.1.3680043.8.498.64119798231341423127851203797783576453",
  "1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487",
  "1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977",
  "1.2.826.0.1.3680043.8.498.13158227087287104386790063637592969032",
  "1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315",
  "1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378",
  "1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409",
  "1.2.826.0.1.3680043.8.498.11148939654362681630811787929668902046",
  "1.2.826.0.1.3680043.8.498.13128506994126383651503283021834372142",
  "1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412",
  "1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598",
  "1.2.826.0.1.3680043.8.498.64663285742626599570990862360902141188",
  "1.2.826.0.1.3680043.8.498.12674069717230126880288353271062347095",
  "1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374"
]
print(f'checking {len(UNKNOWN_SAMPLE_UIDS)} studies previously resolved as unknown')


In [ ]:
from pathlib import Path

TRAIN_SERIES_DIR = f'{COMP_DIR}/train_series'
rows = []
any_tag_found = False

for study_uid in UNKNOWN_SAMPLE_UIDS:
    study_dir = Path(TRAIN_SERIES_DIR) / study_uid
    series_dirs = sorted(p for p in study_dir.iterdir() if p.is_dir())
    for series_dir in series_dirs:
        dcm_files = sorted(series_dir.glob('*.dcm'))
        for dcm_path in dcm_files:
            header = read_laterality_header(dcm_path)
            has_tag = bool(header['ImageLaterality']) or bool(header['Laterality'])
            rows.append({
                'StudyInstanceUID': study_uid,
                'SeriesInstanceUID': series_dir.name,
                'n_instances_in_series': len(dcm_files),
                'has_image_laterality': bool(header['ImageLaterality']),
                'has_laterality': bool(header['Laterality']),
            })
            if has_tag:
                any_tag_found = True
                print(f'FOUND TAG: study={study_uid} series={series_dir.name} file={dcm_path.name} header={header}')

print()
print(f'checked {len(rows)} total instances across {len(UNKNOWN_SAMPLE_UIDS)} studies')
print(f'any instance anywhere carrying ImageLaterality or Laterality: {any_tag_found}')


In [ ]:
import pandas as pd

verify_df = pd.DataFrame(rows)
verify_df.to_csv('/kaggle/working/laterality_verify_sample.csv', index=False)
print('saved laterality_verify_sample.csv to /kaggle/working')
print(verify_df[['has_image_laterality', 'has_laterality']].sum())
